In [1]:
%pip install sqlalchemy==1.3.9
%pip install ipython-sql
%pip install ipython-sql prettytable

%load_ext sql

  Using cached sqlalchemy-1.3.9-cp314-cp314-linux_x86_64.whl
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.51
    Uninstalling SQLAlchemy-2.0.51:
      Successfully uninstalled SQLAlchemy-2.0.51
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.3.9 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
  Using cached sqlalchemy-2.0.51-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.5 kB)
Using cached sqlalchemy-2.0.51-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (3.3 MB)
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 1.3.9
    Uninstalling SQLAlchemy-1.3.9:
      Successfully uninstalled SQLAlchemy-1.3.9
Note: you

In [2]:
## Connect to Database
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [7]:
%pip install -q pandas
%sql sqlite:///my_data1.db

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

Remove blank rows from table

In [10]:
#DROP THE TABLE IF EXISTS

%sql DROP TABLE IF EXISTS SPACEXTABLE;
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///my_data1.db
Done.
 * sqlite:///my_data1.db
Done.


[]

### Tasks
##### Display the names of the unique launch sites  in the space mission

In [62]:
# set(df['Launch_Site'].values)

In [67]:
%%sql
SELECT DISTINCT(Launch_Site) FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


#####  Display 5 records where launch sites begin with the string 'CCA'

In [80]:
%%sql
SELECT * 
FROM SPACEXTABLE 
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


##### Display the total payload mass carried by boosters launched by NASA (CRS)

In [ ]:
%%sql
SELECT SUM(PAYLOAD_MASS__KG_) AS 'payload mass sum for customer NASA (CRS) in kg'
FROM SPACEXTABLE
WHERE Customer LIKE 'NASA (CRS)';

 * sqlite:///my_data1.db
Done.


payload mass sum for customer NASA (CRS)
45596


##### Display average payload mass carried by booster version F9 v1.1

In [ ]:
%%sql
SELECT AVG(PAYLOAD_MASS__KG_) AS 'mean payload mass for booster version F9 v1.1 in kg'
FROM SPACEXTABLE
WHERE Booster_Version LIKE 'F9 v1.1';

 * sqlite:///my_data1.db
Done.


mean payload mass for booster version F9 v1.1 in kg
2928.4


##### List the date when the first succesful landing outcome in ground pad was achieved.


In [ ]:
df[[('ground pad' in value) for value in df['Landing_Outcome'].values]]['Date'].min()

'2015-12-22'

In [107]:
%%sql
SELECT MIN(DATE) 
FROM SPACEXTABLE
WHERE Landing_Outcome LIKE '%ground pad%';

 * sqlite:///my_data1.db
Done.


MIN(DATE)
2015-12-22


##### List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [136]:
%%sql
SELECT DISTINCT(Booster_Version)
FROM SPACEXTABLE
WHERE LOWER(Landing_Outcome) LIKE '%success%drone ship%'
AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


##### List the total number of successful and failure mission outcomes

In [143]:
%%sql
SELECT COUNT(Mission_Outcome) AS 'Total missions'
FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Total missions
101


In [145]:
%%sql
SELECT COUNT(Mission_Outcome) AS 'Successful missions'
FROM SPACEXTABLE
WHERE LOWER(Mission_Outcome) LIKE '%success%';

 * sqlite:///my_data1.db
Done.


Successful missions
100


In [146]:
%%sql
SELECT COUNT(Mission_Outcome) AS 'Failed missions'
FROM SPACEXTABLE
WHERE LOWER(Mission_Outcome) LIKE '%failure%';

 * sqlite:///my_data1.db
Done.


Failed missions
1


##### List all the booster_versions that have carried the maximum payload mass, using a subquery with a suitable aggregate function.

In [158]:
%%sql
SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_ AS 'MAX PAYLOAD_MASS__KG_'
FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);

 * sqlite:///my_data1.db
Done.


Booster_Version,MAX PAYLOAD_MASS__KG_
F9 B5 B1048.4,15600
F9 B5 B1049.4,15600
F9 B5 B1051.3,15600
F9 B5 B1056.4,15600
F9 B5 B1048.5,15600
F9 B5 B1051.4,15600
F9 B5 B1049.5,15600
F9 B5 B1060.2,15600
F9 B5 B1058.3,15600
F9 B5 B1051.6,15600


##### List the records which will display the month names, failure landing_outcomes in drone ship ,booster versions, launch_site for the months in year 2015.

In [175]:
%%sql
SELECT 
CASE SUBSTR(DATE,6,2)
    WHEN '01' THEN 'January' WHEN '02' THEN 'February' WHEN '03' THEN 'March'
    WHEN '04' THEN 'April' WHEN '05' THEN 'May' WHEN '06' THEN 'June'
    WHEN '07' THEN 'July' WHEN '08' THEN 'August' WHEN '09' THEN 'September'
    WHEN '10' THEN 'October' WHEN '11' THEN 'November' WHEN '12' THEN 'December'
  END
AS MONTH, LANDING_OUTCOME, BOOSTER_VERSION, LAUNCH_SITE
FROM SPACEXTABLE 
WHERE SUBSTR(DATE,0,5)='2015'
AND LOWER(LANDING_OUTCOME) LIKE '%failure%ship%';

 * sqlite:///my_data1.db
Done.


MONTH,Landing_Outcome,Booster_Version,Launch_Site
January,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
April,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


##### Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [183]:
%%sql
SELECT LANDING_OUTCOME, COUNT(*) AS outcome_count
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY LANDING_OUTCOME
ORDER BY outcome_count DESC;

 * sqlite:///my_data1.db
Done.


Landing_Outcome,outcome_count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
